ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
 import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA


sns.set_theme(style="whitegrid")
%matplotlib inline

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
 
df = pd.read_csv('CC_GENERAL.csv')


**Check the first five rows of the dataset.**

In [ ]:
 print("--- First 5 Rows ---")
print(df.head())

**Check the shape of the dataset.**

In [ ]:
 print("\n--- Dataset Shape ---")
print(df.shape)


**Check basic information about the dataset using `info()`.**

In [ ]:
 print("\n--- Info ---")
df.info()


**Check summary statistics using `describe()`.**

In [ ]:
 print("\n--- Summary Statistics ---")
print(df.describe())

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
 
df = df.drop('CUST_ID', axis=1)


**Check the missing values in each column.**

In [ ]:
 print("--- Missing Values Before Imputation ---")
print(df.isnull().sum())

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
 f = df.fillna(df.mean())


**Check the missing values again to make sure they were handled.**

In [ ]:
 print("\n--- Missing Values After Imputation ---")
print(df.isnull().sum())

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
 
df.hist(figsize=(15, 12), bins=30, color='skyblue', edgecolor='black')
plt.tight_layout()
plt.show()


**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
 plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
 plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='BALANCE', y='PURCHASES', alpha=0.6, color='purple')
plt.title('Balance vs Purchases')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
 plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='BALANCE', y='CASH_ADVANCE', alpha=0.6, color='coral')
plt.title('Balance vs Cash Advance')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
 
scaler = StandardScaler()


X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
 
inertia_values = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)


**Plot the elbow curve.**

In [ ]:
 plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), inertia_values, marker='o', linestyle='--', color='b')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method Curve')
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
 silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
 plt.figure(figsize=(8, 4))
plt.plot(k_range, silhouette_scores, marker='s', linestyle='-', color='g')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score Curve')
plt.show()


**Create a table showing each K value and its silhouette score.**

In [ ]:
 silhouette_df = pd.DataFrame({
    'K_Value': list(k_range),
    'Silhouette_Score': silhouette_scores
})
print("--- Silhouette Scores Table ---")
print(silhouette_df.to_string(index=False))

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_k = 4 

final_kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
final_labels = final_kmeans.fit_predict(X_scaled)



**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
 df['Cluster'] = final_labels

**Check the first five rows after adding the cluster labels.**

In [ ]:
 print("--- First 5 Rows with Cluster Labels ---")
print(df.head())

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
print("--- Cluster Feature Means Summary ---")
print(cluster_summary)



**Check how many customers are in each cluster.**

In [ ]:
print("\n--- Customer Counts per Cluster ---")
print(df['Cluster'].value_counts().sort_index()) 

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
 # Reduce dimensions to 2 components using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Create temporary DataFrame for visualization
pca_df = pd.DataFrame(data=X_pca, columns=['PCA1', 'PCA2'])
pca_df['Cluster'] = final_labels

# Plot the 2D clusters
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pca_df, 
    x='PCA1', 
    y='PCA2', 
    hue='Cluster', 
    palette='Set1', 
    alpha=0.7, 
    edgecolor='none'
)
plt.title('Final Customer Clusters Visualized via 2D PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?
It is an unsupervised learning problem because the dataset does not contain any predefined target labels or ground truth categories. We are not predicting a specific outcome (like fraud vs. non-fraud). Instead, the goal is to discover underlying, natural patterns and group similarities among customers based solely on their transaction behaviors.

2. Why did we remove the `CUST_ID` column?
The CUST_ID column contains unique text identifiers for each customer. It is not a behavioral metric. Because K-Means uses mathematical distances to group data points, leaving a unique, non-numeric, or sequential identifier in the dataset would distort distance calculations and introduce noise without adding any predictive or clustering value.

3. Which columns had missing values?
In the standard CC_GENERAL.csv dataset, two columns contain missing values:

MINIMUM_PAYMENTS: Missing 313 values.

CREDIT_LIMIT: Missing 1 value.

4. How did you handle the missing values?
Per the assignment instructions, missing values were handled using mean imputation. The null values in MINIMUM_PAYMENTS and CREDIT_LIMIT were calculated and replaced with the average (mean) value of their respective columns.

5. Why is scaling important before applying K-Means?
K-Means relies heavily on Euclidean distance to determine cluster boundaries. Features in this dataset have vastly different scales—for example, account balances and credit limits can be in the thousands of dollars, whereas frequencies (like PURCHASES_FREQUENCY) range strictly between 0 and 1. Without scaling, features with massive absolute numbers would completely dominate the distance math, rendering the frequency metrics mathematically irrelevant. StandardScaler standardizes them to have a mean of 0 and variance of 1, giving every feature an equal voice.

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.
We chose $K = 4$. * Elbow Method: In the inertia plot, the curve drops steeply from $K=1$ to $K=3$, and the rate of decrease noticeably slows down ("elbows") around $K=3$ or $K=4$. This indicates a point of diminishing returns for adding more clusters.Silhouette Score: While lower clusters ($K=2$ or $K=3$) often yield marginally higher mathematical silhouette scores due to simpler geometry, $K=4$ offers a strong compromise—maintaining clean cluster separation while yielding much more distinct, actionable behavioral segments for a real-world business case.

7. Based on the cluster summary table, describe each customer segment in your own words.
Cluster 0: The Carefree Transactors / Every-day Users. These are highly active card users with moderate balances. They make frequent purchases (especially installments) and rarely take out cash advances. They use their card as a primary payment tool.

Cluster 1: The Cash Advance Reliers. These customers maintain high balances but rarely use their cards for shopping. Instead, they use their credit cards like an ATM, pulling out frequent and substantial cash advances.

Cluster 2: The VIP / High-Spenders. A small but powerful group. These customers have massive credit limits, maintain high balances, and make enormous purchases (both one-off luxury buys and frequent installments).

Cluster 3: The Stagnant / Frugal Cardholders. This group features low balances, low credit limits, and very minimal transactional activity. They keep the card open but rarely use it for purchases or cash advances.

8. Which cluster may represent high-value customers?
Cluster 2 (The VIP / High-Spenders) represents the highest-value customers. They hold the highest credit limits and generate massive transaction volumes, which directly translates to substantial interchange fee revenue for the credit card company.
9. Which cluster may represent customers who rely more on cash advance?
Cluster 1 (The Cash Advance Reliers) represents this group. They display disproportionately high values for CASH_ADVANCE and CASH_ADVANCE_FREQUENCY relative to their purchase behavior

10. How can a company use these clusters for marketing strategy?
For High-Spenders (Cluster 2): Offer premium rewards, exclusive cashback tiers, invitations to luxury events, and higher credit limits to encourage continued high-volume spending.

For Cash Advance Users (Cluster 1): Target them with promotional offers for structured personal loans or lower interest rate balance transfers, as they are actively looking for liquid capital.

For Every-day Users (Cluster 0): Co-market with retail or grocery partners to offer point-multipliers on daily transactions or zero-interest installment plans on electronics.

For Inactive Users (Cluster 3): Launch "We Miss You" reactivation campaigns offering temporary statement credits or introductory perks if they hit a baseline spending threshold within 30 days.